# Resume ISLES'24 NCCT Fold 4 on Kaggle — sharded handoff

Before running:

1. Create a Kaggle Notebook with **GPU T4 x2**.
2. Turn **Internet ON**.
3. Attach the private Dataset `isles2024-ncct-fold4-resume-shards`.
4. Run all cells.

Only one T4 is exposed to nnU-Net so the transferred Fold 4 checkpoint keeps single-GPU continuation semantics.


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable,'-m','pip','install','-q','nnunetv2==2.8.1'],check=True)
REPO=Path('/kaggle/working/HybridStrokeSeg-Reproduction')
if REPO.exists():
    import shutil; shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--depth','1',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction.git',
    str(REPO)
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)


## Locate the private sharded input


In [ ]:
matches=list(Path('/kaggle/input').rglob('handoff_manifest.json'))
print('Manifest matches:',matches)
if len(matches)!=1:
    raise RuntimeError('Attach exactly one sharded Fold 4 resume Dataset input')
INPUT_DIR=matches[0].parent
print('Using input directory:',INPUT_DIR)


## Verify GPU and resume Fold 4


In [ ]:
import torch
print('Kaggle GPUs:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i,torch.cuda.get_device_name(i))

env=os.environ.copy()
env['CUDA_VISIBLE_DEVICES']='0'
subprocess.run([
    sys.executable,
    str(REPO/'scripts'/'run_kaggle_fold4_shards.py'),
    '--input-dir',str(INPUT_DIR),
    '--work-root','/kaggle/working/isles2024_runtime',
    '--output-root','/kaggle/working/isles2024_fold4_output'
],check=True,env=env)


## Final output

When training finishes, Kaggle saves `/kaggle/working/isles2024_fold4_output` as notebook output. Keep that output private.


In [ ]:
OUT=Path('/kaggle/working/isles2024_fold4_output')
print('Output exists:',OUT.exists())
if OUT.exists():
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            print(p.relative_to(OUT),round(p.stat().st_size/(1024**2),2),'MiB')
